In [1]:
!pwd

/users/khordadi/truejit/evaluation/compilation-plan


In [2]:
import os
import sys
from pathlib import Path
from solver import *
from profiling import *
import numpy as np

In [3]:
benchmark = ffmpeg
for wl in benchmark.workloads:
    print(wl.name)

mov-to-mp4
mp4-to-mkv
mp4-to-wav
mp4-resize-720p
jpegs-to-mp4
mp4-trim
mp4-flip-vertical
mp4s-concat
mp4-stream-copy
mp4-to-pngs
mp4-speedup-2x
mp4-slowdown-2x
mp4-mono-audio
mp4-remove-audio
mp4-to-flac
mp4-gaussian-blur


In [4]:
# generate_static_info(benchmark.binary)

In [5]:
static_info = get_static_info(benchmark.binary)
static_info

,id,name,size.bytecode,size.static
0,24,__wasm_call_ctors,8,864
1,25,undefined_weak:__wasilibc_find_relpath_alloc,4,504
2,26,_start,69,984
3,27,__SIG_IGN,3,512
4,28,__SIG_ERR,5,504
...,...,...,...,...
27485,27509,ff_tx_fft16_ns_int32_c,2188,3344
27486,27510,ff_tx_fft8_ns_int32_c,1144,2096
27487,27511,ff_tx_fft4_ns_int32_c,382,912
27488,27512,ff_tx_fft2_ns_int32_c,163,664


In [6]:
wl = benchmark.workloads[0]

In [7]:
profile = pd.read_csv(profiles_root(benchmark.binary, wl.name) / 'profile.csv')
profile

,id,name,size.bytecode,size.static,start.jit,exec.jit,freq.jit,compilation.jit,size.dynamic.jit,start.interp,exec.interp,freq.interp,exec.spec,compilation.spec,size.dynamic.spec
0,26,_start,69,984,70848,0,0,3675552,984,72965,0,0,999999999999,999999999999,999999999999
1,24,__wasm_call_ctors,8,864,3747522,0,1,1823296,864,77094,0,1,999999999999,999999999999,999999999999
2,30,init,20,840,5571401,0,1,1978378,840,77805,0,1,999999999999,999999999999,999999999999
3,1540,__wasi_clock_time_get,20,824,7550469,0,109,1956529,824,79312,0,109,999999999999,999999999999,999999999999
4,1588,__wasilibc_populate_preopens,182,1296,9513100,0,1,5488068,1296,87298,0,1,999999999999,999999999999,999999999999
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2455,2384,dummy,3,512,114039339266,0,1,1467188,512,11307168754840,0,1,999999999999,999999999999,999999999999
2456,1597,__wasm_call_dtors,15,864,114040809792,1000000,1,1879842,864,11307168757371,0,1,999999999999,999999999999,999999999999
2457,1596,dummy,3,512,114042691897,0,1,1449014,512,11307168758076,0,1,999999999999,999999999999,999999999999
2458,1715,__stdio_exit,389,1712,114044142365,0,1,8313526,1712,11307168758729,0,1,999999999999,999999999999,999999999999


In [20]:
plans_names = [
    'Interpreter', 
    'JIT', 
    'AOT', 
    'Background JIT',

    'Hybrid (AOT + JIT)', 
    'Hybrid (Interpreter + JIT)', 
    'Hybrid (AOT + Interpreter)',
    
    'CAPER (Unconstrained)',
    'CAPER (Constrained)',
    ]

plans = []
plans.append(Planner(None, None, 'interpret').plan(static_info, profile)) # Interpreter
plans.append(Planner(None, None, 'jit').plan(static_info, profile)) # JIT
plans.append(Planner(None, None, 'static').plan(static_info, profile)) # AOT
plans.append(Planner(None, None, 'async').plan(static_info, profile)) # Background JIT

plans.append(Planner(EndToEndTime(), None, 'aot').plan(static_info, profile)) # Hybrid (AOT + JIT)
plans.append(Planner(EndToEndTime(), [Constraint(StaticCodeSize(), upper_bound=0)], 'hybrid').plan(static_info, profile)) # Hybrid (Interpreter + JIT)
plans.append(Planner(EndToEndTime(), [Constraint(StaticCodeSize(), upper_bound=1_500_000), Constraint(DynamicCodeSize(), upper_bound=1_500_000)], 'interpreter').plan(static_info, profile)) # Hybrid (AOT + Interpreter)

plans.append(Planner(EndToEndTime(), None, 'aot').plan(static_info, profile)) # CAPER (Unconstrained)
plans.append(Planner(EndToEndTime(), [Constraint(StaticCodeSize(), upper_bound=1_000_000)], 'jit').plan(static_info, profile)) # CAPER (Constrained)

history = get_history(benchmark.binary, wl.name)

evaluations = []
for i, plan in enumerate(plans):
    print(f"[plan={plans_names[i]}] {plan.shape[0]} rows")
    evaluation = PlanEvaluation(static_info, profile, plan, history)
    evaluation.print()
    evaluations.append(evaluation)

[goal] EndToEndTime
Solving...
[status] Optimal
[elapsed time] 0.16 seconds
[goal] EndToEndTime
[constraint] StaticCodeSize <= 0
Solving...
[status] Optimal
[elapsed time] 0.13 seconds
[goal] EndToEndTime
[constraint] StaticCodeSize <= 1500000
[constraint] DynamicCodeSize <= 1500000
Solving...
[status] Optimal
[elapsed time] 1.24 seconds
[goal] EndToEndTime
Solving...
[status] Optimal
[elapsed time] 0.16 seconds
[goal] EndToEndTime
[constraint] StaticCodeSize <= 1000000
Solving...
[status] Optimal
[elapsed time] 0.42 seconds
[plan=Interpreter] 27490 rows
[plan evaluation] merged static_info | base_profile: 27490 rows
[plan evaluation] merged static_info | base_profile | plan: 27490 rows
{'jit': 0, 'interpret': 27490, 'async_compilation': 0, 'specialize': 0, 'static': 0, 'e2e': np.float64(11209155000000.0), 'exec': np.float64(11209155000000.0), 'waiting': 0, 'compilation': 0, 'dynamic_code_size': 0, 'static_code_size': 0}
[plan=JIT] 27490 rows
[plan evaluation] merged static_info | base

In [21]:
# plans_names = [
#     'Interpreter', 
#     'JIT', 
#     'AOT', 
#     'Background JIT',
#     'Hybrid (AOT + JIT)', 
#     'Hybrid (Interpreter + JIT)', 
#     'CAPER'
#     ]

# plans = []
# plans.append(Planner(None, None, 'interpret').plan(static_info, profile))
# plans.append(Planner(None, None, 'jit').plan(static_info, profile))
# plans.append(Planner(None, None, 'static').plan(static_info, profile))
# plans.append(Planner(None, None, 'async').plan(static_info, profile))

# all_ids = static_info['id'].tolist()
# plan = plan_json_to_df(plans_root(benchmark.binary) / 'heuristic.aot-vs-jit.json')
# unplanned_ids = set(all_ids) - set(plan['id'].tolist())
# defaults = pd.DataFrame({'id': list(unplanned_ids), 'mode': ['jit'] * len(unplanned_ids)})
# plan = pd.concat([plan, defaults])
# plans.append(plan)

# plan = plan_json_to_df(plans_root(benchmark.binary) / 'heuristic.interp-vs-jit.json')
# unplanned_ids = set(all_ids) - set(plan['id'].tolist())
# defaults = pd.DataFrame({'id': list(unplanned_ids), 'mode': ['jit'] * len(unplanned_ids)})
# plan = pd.concat([plan, defaults])
# plans.append(plan)


# plans.append(Planner(EndToEndTime(), [Constraint(StaticCodeSize(), upper_bound=1_000_000)], 'jit').plan(static_info, profile))

# history = get_history(benchmark.binary, wl.name)

# evaluations = []
# for i, plan in enumerate(plans):
#     print(f"[plan={plans_names[i]}] {plan.shape[0]} rows")
#     evaluation = PlanEvaluation(static_info, profile, plan, history)
#     evaluations.append(evaluation)

In [22]:
data =[]
for e in evaluations:
    data.append((e.e2e, e.exec, e.waiting, e.static_code_size, e.compilation + e.exec))

df = pd.DataFrame(data, columns=['e2e', 'exec', 'waiting', 'storage_overhead', 'cpu'])
df['plan'] = plans_names
# make times to s instead of ns
df['e2e'] /= 1e9
df['exec'] /= 1e9
df['waiting'] /= 1e9
df[['plan', 'waiting', 'storage_overhead', 'e2e', 'cpu']]

,plan,waiting,storage_overhead,e2e,cpu
0,Interpreter,0.000000,0,11209.155000,1.120916e+13
1,JIT,38.801161,0,113.472161,1.134722e+11
2,AOT,0.000000,55138560,74.671000,7.467100e+10
3,Background JIT,19.400581,0,94.071581,1.134722e+11
4,Hybrid (AOT + JIT),0.000000,5028864,74.646000,7.464600e+10
5,Hybrid (Interpreter + JIT),7.403663,0,83.280663,9.068433e+10
6,Hybrid (AOT + Interpreter),0.000000,1499880,75.039000,7.503900e+10
7,CAPER (Unconstrained),0.000000,5028864,74.646000,7.464600e+10
8,CAPER (Constrained),0.759011,1000000,76.581011,7.734002e+10


In [23]:
print(df[['plan', 'waiting', 'storage_overhead', 'e2e', 'cpu']].to_string(index=False))

                      plan   waiting  storage_overhead          e2e          cpu
               Interpreter  0.000000                 0 11209.155000 1.120916e+13
                       JIT 38.801161                 0   113.472161 1.134722e+11
                       AOT  0.000000          55138560    74.671000 7.467100e+10
            Background JIT 19.400581                 0    94.071581 1.134722e+11
        Hybrid (AOT + JIT)  0.000000           5028864    74.646000 7.464600e+10
Hybrid (Interpreter + JIT)  7.403663                 0    83.280663 9.068433e+10
Hybrid (AOT + Interpreter)  0.000000           1499880    75.039000 7.503900e+10
     CAPER (Unconstrained)  0.000000           5028864    74.646000 7.464600e+10
       CAPER (Constrained)  0.759011           1000000    76.581011 7.734002e+10
